# 🎬 Keep One Person
Remove people from a video as if they had never been there - **keep just one person** (everyone else goes), or **remove the people you choose** (everyone else stays). The sound stays as it was.

1. Press the **▶ play button** below. Click **Run anyway** if asked, then **allow Google Drive** (it saves your progress and finished videos).
2. When the app appears below, **upload your video** right away (and, for *keep one person*, a few **photos of that person**) and press **Next** - the AI models finish downloading meanwhile.
3. Follow the app. (You can also put files in **Google Drive › Keep One Person** - the app then offers them.)

Finished videos appear in **Google Drive › Keep One Person › Finished**. This notebook uses an **A100 GPU** (Colab Pro); if none is free, choose another under *Runtime › Change runtime type*.

In [ ]:
#@title ▶ Press this play button - the app appears below when it is ready
#@markdown The app opens below this box (and as a link ending in gradio.live). Keep this tab open while it works.
SAVE_TO_GOOGLE_DRIVE = True  #@param {type:"boolean"}
import os, subprocess, sys, importlib, time, threading
T0 = time.time()
JOBS = []

def bg(title, cmd, required=True, skip=False):
    # setup steps run at the same time (downloads, installs) to save minutes
    if skip:
        return
    JOBS.append((title, subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                                         stderr=subprocess.STDOUT, text=True), required))

def wait_all(only=None):
    for title, p, required in [j for j in JOBS if only is None or j[0] in only]:
        out = p.communicate()[0] or ""
        if p.returncode != 0:
            print(out[-1500:])
            if required:
                raise RuntimeError("Setup step failed: " + title + ". Please send a screenshot of this box.")
            print("⚠️ " + title + " failed - continuing without it.")
        else:
            print(f"✅ {title} ({time.time() - T0:.0f} s)", flush=True)
    JOBS[:] = [j for j in JOBS if not (only is None or j[0] in only)]

# 1) GPU
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU is connected. Menu: Runtime > Change runtime type > A100 GPU "
                       "(or L4 / T4) > Save, then press the play button again.")
print("✅ GPU: " + torch.cuda.get_device_name(0), flush=True)

# 2) everything is downloaded and installed at the same time
CODE, R = "/content/keep_one_person", "/content/repos"
os.makedirs(R + "/sam2_ckpt", exist_ok=True)
os.chdir("/content")
bg("Program", f"git -C {CODE} fetch -q --depth 1 origin keep-one-person && git -C {CODE} reset -q --hard FETCH_HEAD"
   if os.path.isdir(CODE + "/.git") else f"git clone -q --depth 1 -b keep-one-person https://github.com/grok69420/minimal-launcher.git {CODE}")
bg("Tracking engine", f"git clone -q --depth 1 https://github.com/facebookresearch/sam2.git {R}/sam2",
   skip=os.path.isdir(R + "/sam2/sam2"))
# (insightface pulls in the CPU onnxruntime, which hides the GPU one: it is removed afterwards)
# (uv installs the same packages several times faster than pip; pip is the fallback)
PK = ("hydra-core iopath av addict einops future timm yapf imageio-ffmpeg ultralytics==8.4.194 "
      "gradio==6.31.0 diffusers==0.42.1")
bg("Packages", f"(uv pip install --system -q {PK} || pip -q install {PK}) && "
   "(uv pip install --system -q insightface==2.1 || pip -q install insightface==2.1 || true) && "
   "(uv pip uninstall --system -q onnxruntime || pip -q uninstall -y onnxruntime || true) && "
   "(uv pip install --system -q --reinstall --no-deps onnxruntime-gpu || "
   "pip -q install --force-reinstall --no-deps onnxruntime-gpu) && touch /content/.kop_packages_v8",
   skip=os.path.exists("/content/.kop_packages_v8"))
bg("Painting engine", f"git clone -q --depth 1 https://github.com/zibojia/MiniMax-Remover.git {R}/MiniMax-Remover",
   required=False, skip=os.path.isdir(R + "/MiniMax-Remover"))
bg("Painting model (2.5 GB)", "python3 -c \"from huggingface_hub import snapshot_download as d; "
   f"d(repo_id='zibojia/minimax-remover', local_dir='{R}/minimax_weights', "
   "allow_patterns=['vae/*', 'transformer/*', 'scheduler/*'])\"", required=False,
   skip=os.path.exists(R + "/minimax_weights/transformer"))
SAM = "https://dl.fbaipublicfiles.com/segment_anything_2/092824/"
bg("Tracking models", f"wget -q -nc -P {R}/sam2_ckpt {SAM}sam2.1_hiera_base_plus.pt {SAM}sam2.1_hiera_large.pt")
YOLO = "https://github.com/ultralytics/assets/releases/download/v8.4.0/"
bg("Detection models", f"wget -q -nc {YOLO}yolo11x-seg.pt {YOLO}yolo11x.pt {YOLO}yolo11s-cls.pt")
bg("Fast decoder", f"mkdir -p {R}/taehv && cd {R}/taehv && for f in taehv.py taew2_1.pth; do "
   "test -s $f || wget -q -O $f https://github.com/madebyollin/taehv/raw/main/$f; done", required=False)
bg("Face model", "mkdir -p ~/.insightface/models && cd ~/.insightface/models && "
   "(test -d buffalo_l || (wget -q https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_l.zip "
   "&& unzip -q -o buffalo_l.zip -d buffalo_l && rm buffalo_l.zip))", required=False)
bg("Spare background engine", f"(test -d {R}/ProPainter/model || git clone -q --depth 1 "
   f"https://github.com/sczhou/ProPainter.git {R}/ProPainter) && cd {R}/ProPainter && mkdir -p weights && "
   "for f in ProPainter.pth raft-things.pth recurrent_flow_completion.pth; do wget -q -nc -P weights "
   "https://github.com/sczhou/ProPainter/releases/download/v0.1.0/$f; done", required=False)
print("… Downloading and installing", flush=True)

# 3) Google Drive (saves progress of long videos) - while the downloads run
if SAVE_TO_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        os.makedirs("/content/drive/MyDrive/Keep One Person", exist_ok=True)
        print("✅ Google Drive connected (finished videos: Keep One Person/Finished).")
    except Exception as e:
        print("⚠️ Google Drive not connected - download the finished video from the app. (", e, ")")
# 4) the app opens as soon as the program and packages are in; the AI models keep downloading
#    while you upload (the video scan starts as soon as they are there)
wait_all(["Program", "Packages"])
if CODE not in sys.path:
    sys.path.insert(0, CODE)
import remove_people, app
importlib.reload(remove_people)
importlib.reload(app)
app.SETUP.clear()

def finish_setup():
    try:
        wait_all()
        os.makedirs(R + "/sam2/checkpoints", exist_ok=True)
        for f in os.listdir(R + "/sam2_ckpt"):
            if not os.path.exists(f"{R}/sam2/checkpoints/{f}"):
                os.replace(f"{R}/sam2_ckpt/{f}", f"{R}/sam2/checkpoints/{f}")
        print(f"✅ All AI models ready after {time.time() - T0:.0f} s.", flush=True)
        # film quality engine (~8 GB, about a minute): fetched now in the background, used only
        # when "Film quality" is chosen
        subprocess.Popen(f"flock {R}/.svor_setup.lock bash {CODE}/tools/setup_svor.sh > /content/svor_setup.log "
                         f"2>&1 && touch {R}/.svor_ready", shell=True)
    except Exception as e:
        app.SETUP_ERR.append(str(e))
        print("⚠️", e, flush=True)
    finally:
        app.SETUP.set()
    app.preload()
threading.Thread(target=finish_setup, daemon=True).start()
print(f"✅ App ready after {time.time() - T0:.0f} s - upload your video now (the AI models finish "
      "downloading meanwhile). Use the app below, or open the link ending in gradio.live.", flush=True)
app.demo.queue().launch(share=True, debug=True, inline=True, height=1000, css=app.CSS,
                        allowed_paths=[app.CACHE, "/content/drive"])